In [1]:
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

service = Service("C:/Users/sippa/Downloads/Work/chromedriver/chromedriver.exe")
driver = webdriver.Chrome(service=service)

In [ ]:
email = "forgptkung@gmail.com"
password = "ForStockscreener12"
column = "Technical Rating"

# Navigate to screener
driver.get("https://www.tradingview.com/screener/")
time.sleep(3)

# Add column
addcolumn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-qa-id='screener-add-column-button']"))
)
addcolumn.click()

searchcolumn = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[aria-label='Type column name']"))
)
searchcolumn.send_keys(column)
time.sleep(2)

technicalrating = driver.find_element(By.CLASS_NAME, "highlighted-xWsxD6Lf")
technicalrating.click()
time.sleep(1)

adddaycolumn = driver.find_element(By.CSS_SELECTOR, "button[data-overflow-tooltip-text='Add column']")
adddaycolumn.click()
time.sleep(2)

# Sign in
try:
    emailbutton = driver.find_element(By.CSS_SELECTOR, "button[name='Email']")
    emailbutton.click()
    
    usernameinput = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "id_username"))
    )
    usernameinput.send_keys(email)
    
    passwordinput = driver.find_element(By.ID, "id_password")
    passwordinput.send_keys(password)
    
    signinbutton = driver.find_element(By.CSS_SELECTOR, "button[data-overflow-tooltip-text='Sign in']")
    signinbutton.click()
    
    print("Signed in successfully!")
except Exception as e:
    print(f"Sign in error: {e}")

# Wait for page to fully load
time.sleep(5)

Signed in successfully!


In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from io import StringIO

# Wait for page to load after sign in
time.sleep(5)

# Find the scrollable container in the lower half
try:
    scrollable = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.shadow-zuRb9wy5"))
    )
except:
    try:
        scrollable = driver.find_element(By.CLASS_NAME, "wrap-vSb6C0Bj")
    except:
        scrollable = driver.find_element(By.TAG_NAME, "table")

print("Starting data load...")
print(f"Scrollable element found: {scrollable.tag_name}")

last_row_count = 0
no_change_count = 0
scroll_attempts = 0
loaded_stocks = set()

while scroll_attempts < 1000:
    # Scroll from the scrollable element origin (lower half)
    scroll_origin = ScrollOrigin.from_element(scrollable)
    ActionChains(driver)\
        .scroll_from_origin(scroll_origin, 0, 20000)\
        .perform()
    
    # Wait 2 seconds after scrolling
    time.sleep(2)
    
    # Check row count every 5 scrolls
    if scroll_attempts % 5 == 0:
        try:
            temp_df = pd.read_html(StringIO(driver.page_source))
            current_row_count = len(temp_df[0])
            
            # Print newly loaded stock names
            ticker_col = 'Ticker' if 'Ticker' in temp_df[0].columns else 'Symbol'
            if ticker_col in temp_df[0].columns:
                current_stocks = set(temp_df[0][ticker_col].tolist())
                new_stocks = current_stocks - loaded_stocks
                if new_stocks:
                    print(f"\n--- Newly loaded stocks ({len(new_stocks)} new): ---")
                    for stock in sorted(list(new_stocks)[:10]):
                        print(stock)
                    if len(new_stocks) > 10:
                        print(f"... and {len(new_stocks) - 10} more")
                    loaded_stocks = current_stocks
            
            print(f"Total loaded: {current_row_count} rows (scroll: {scroll_attempts})")
            
            # Check if loading is complete
            if current_row_count == last_row_count:
                no_change_count += 1
                print(f"No change: {no_change_count}/5")
                if no_change_count >= 5:
                    print(f"\n✓ Data loading complete! Total rows: {current_row_count}")
                    break
            else:
                no_change_count = 0
                last_row_count = current_row_count
                
        except Exception as e:
            print(f"Error reading data: {e}")
    
    scroll_attempts += 1

# Get final data
data = driver.page_source
data_df = pd.read_html(StringIO(data))
print(f"\nFinal DataFrame shape: {data_df[0].shape}")
print(f"Total unique stocks loaded: {len(loaded_stocks)}")
print(f"Columns: {list(data_df[0].columns)}")
data_df[0]

Starting data load...
Scrollable element found: div

--- Newly loaded stocks (800 new): ---
CTASCintas Corporation
DLRDigital Realty Trust, Inc.REIT
EWEdwards Lifesciences Corporation
FFord Motor Company
FNFabrinet
GDGeneral Dynamics Corporation
NDAQNasdaq, Inc.
VLTOVeralto Corp
WDCWestern Digital Corporation
WYWeyerhaeuser CompanyREIT
... and 790 more
Total loaded: 800 rows (scroll: 0)

--- Newly loaded stocks (500 new): ---
ACTEnact Holdings, Inc.
ALGMAllegro MicroSystems, Inc.
ALTBAlpine Auto Brokers IncD
AWIArmstrong World Industries Inc
CHRDChord Energy Corporation
HSICHenry Schein, Inc.
HYMCHycroft Mining Holding Corporation
MCMoelis & Company
OZKBank OZK
STAGSTAG Industrial, Inc.REIT
... and 490 more
Total loaded: 1300 rows (scroll: 5)

--- Newly loaded stocks (500 new): ---
ARCBArcBest Corporation
CRMLCritical Metals Corp.
DXPEDXP Enterprises, Inc.
GFFGriffon Corporation
IHSIHS Holding Limited
MANUManchester
NADNuveen Quality Municipal Income Fund of Benef.InterestCEF
PCTPureCy

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=59776): Read timed out. (read timeout=120)